# Prepare the dataset

Load the examples, normalize them for Gemma, and write reproducible training and evaluation splits.

In [1]:
import json
from pathlib import Path

from datasets import Dataset
import ipywidgets as widgets
from IPython.display import clear_output, display
from transformers import AutoProcessor

MODEL_ID = 'google/gemma-4-E4B-it'
DATASET_DIR = Path('dataset')
TRAIN_PATH = DATASET_DIR / Path('train.jsonl')
EVALUATE_PATH = DATASET_DIR / Path('evaluate.jsonl')

processor = AutoProcessor.from_pretrained(MODEL_ID)

In [2]:
source_paths = sorted(DATASET_DIR.glob('*.json'), key=lambda path: int(path.stem.split('-', 1)[0]))
examples = {path.stem: json.loads(path.read_text()) for path in source_paths}

In [3]:
SYSTEM_COLOR = '34'
USER_COLOR = '34'
ASSISTANT_COLOR = '32'
FIELD_COLOR = '38;5;245'

selector = widgets.Dropdown(
    options=sorted(examples, key=lambda name: int(name.split('-', 1)[0])),
    description='Example:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='720px'),
)
transcript = widgets.Output(
    layout=widgets.Layout(
        width='720px',
        min_height='340px',
        border='1px solid',
        padding='12px',
        overflow='auto',
    )
)


def render(name):
    items = examples[name]['input']

    with transcript:
        clear_output(wait=True)
        for item in items:
            if item['type'] == 'message':
                role = 'system' if item['role'] == 'developer' else item['role']
                content = item['content']
                text = content if isinstance(content, str) else ''.join(part['text'] for part in content)
                color = {'system': SYSTEM_COLOR, 'user': USER_COLOR, 'assistant': ASSISTANT_COLOR}[role]
                print(f'\033[{color}m[{role}]\033[0m\n{text}\n')
            elif item['type'] == 'function_call':
                print(f"\033[{ASSISTANT_COLOR}m[assistant -> {item['name']}()]\033[0m")
                arguments = json.loads(item['arguments'])
                
                for name, value in arguments.items():
                    print(f"\033[{FIELD_COLOR}m{name}:\033[0m {value}")
                print()
            elif item['type'] == 'function_call_output':
                print(f'\033[{ASSISTANT_COLOR}m[tool result]\033[0m')
                result = json.loads(item['output'])
                
                for name, value in result.items():
                    if name == 'output':
                        print(f"\033[{FIELD_COLOR}m{name}:\033[0m\n{value}")
                    else:
                        print(f"\033[{FIELD_COLOR}m{name}:\033[0m {value}")
                print()


selector.observe(lambda change: render(change['new']), names='value')
render(selector.value)
display(widgets.VBox([selector, transcript], layout=widgets.Layout(gap='12px')))

## Normalize

The source files use the Responses API shape. Convert them to the Transformers chat format expected by Gemma's bundled chat template.

In [4]:
def to_transformers_tools(tools):
    return [{
        'type': 'function',
        'function': {
            key: value
            for key, value in tool.items()
            if key not in {'type', 'strict'}
        },
    } for tool in tools]


def to_transformers_message(item, tool_names):
    match item['type']:
        case 'message' if item['role'] == 'developer':
            return None
        case 'message':
            content = item['content']
            return {
                'role': item['role'],
                'content': content if isinstance(content, str) else ''.join(
                    part['text'] for part in content if part['type'] == 'input_text'
                ),
            }
        case 'function_call':
            tool_names[item['call_id']] = item['name']
            return {
                'role': 'assistant',
                'content': '',
                'tool_calls': [{
                    'type': 'function',
                    'id': item['call_id'],
                    'function': {
                        'name': item['name'],
                        'arguments': json.loads(item['arguments']),
                    },
                }],
            }
        case 'function_call_output':
            return {
                'role': 'tool',
                'name': tool_names[item['call_id']],
                'tool_call_id': item['call_id'],
                'content': item['output'],
            }
        case _:
            raise ValueError(f"Unsupported input type: {item['type']!r}")


def to_transformers_conversation(example):
    tool_names = {}
    messages = [
        message
        for item in example['input']
        if (message := to_transformers_message(item, tool_names)) is not None
    ]
    return {'messages': messages, 'tools': to_transformers_tools(example['tools'])}


normalized_examples = [to_transformers_conversation(examples[path.stem]) for path in source_paths]
assert len(normalized_examples) == len(source_paths) == 177

## Check

Confirm that every example has a valid chat-template representation before splitting the dataset.

In [5]:
for example in normalized_examples:
    assert set(example) == {'messages', 'tools'}
    assert all(message['role'] not in {'system', 'developer'} for message in example['messages'])
    
    seen_call_ids = set()
    
    for message in example['messages']:
        for tool_call in message.get('tool_calls', []):
            assert isinstance(tool_call['function']['arguments'], dict)
            seen_call_ids.add(tool_call['id'])
        if message['role'] == 'tool':
            assert message['tool_call_id'] in seen_call_ids
    
    rendered = processor.apply_chat_template(
        example['messages'],
        tools=example['tools'],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )
    
    assert isinstance(rendered, str) and rendered

# Split

Create a 85/15 train/evaluation split, then write each split as `jsonl`.

In [6]:
dataset = Dataset.from_list(normalized_examples)
splits = dataset.train_test_split(test_size=0.15, seed=42, shuffle=True)
train = splits['train']
evaluate = splits['test']

assert len(train) == 150
assert len(evaluate) == 27
assert train.column_names == evaluate.column_names == ['messages', 'tools']

train.to_json(TRAIN_PATH, orient='records', lines=True, force_ascii=False)
evaluate.to_json(EVALUATE_PATH, orient='records', lines=True, force_ascii=False)

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

36145